# 4. Modelo SARIMAX
El modelo SARIMAX (Seasonal ARIMA with eXogenous regressors) es una extensión del modelo SARIMA que incorpora, además de los componentes autorregresivos, de media móvil y estacionales, la posibilidad de incluir variables exógenas o predictoras externas. Se expresa como:

SARIMAX(p,d,q)(P,D,Q,s)

donde:

(p,d,q) representan los componentes no estacionales y (P,D,Q) representan los componentes estacionales.

s indica el periodo de la estacionalidad (por ejemplo, 24 para datos horarios diarios).

y adicionalmente, X representa el conjunto de variables exógenas que se incorporan al modelo.

La principal ventaja de SARIMAX frente a SARIMA es su capacidad para mejorar la predicción al incorporar información adicional relevante, como pueden ser variables temporales (día de la semana, mes...) u otras variables explicativas que influyen en la serie objetivo.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_squared_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox


# =======================================================================================
# LECTURA DEL SUPERDATASET (de precios horarios de la electricidad, generacion y climatología)
# =======================================================================================
df_dummy = pd.read_csv('../../../Dataset_Unificado/Dataset_Unificado1.csv', sep=';')
df_dummy['datetime'] = pd.to_datetime(df_dummy['datetime'])
df_dummy.set_index('datetime', inplace=True) 
# A partir de aquí, 'datetime' ya no es una columna

# =======================================================================================
# 0. DESPLAZAMIENTO DE VARIABLES (SHIFT 24H) PARA EVITAR DATA LEAKAGE
# =======================================================================================
# 1. Identificamos qué columnas NO se deben desplazar (las del calendario actual y el precio a predecir)
cols_temporales = ['year', 'month', 'day', 'hour', 'dayofweek', 'is_weekend']
target = 'price'

# 2. Identificamos las exógenas (todas las demás: demanda, generación, clima...)
cols_exogenas = [col for col in df_dummy.columns if col not in cols_temporales and col != target]

# 3. Desplazamos las variables exógenas 24 posiciones (24 horas)
df_dummy[cols_exogenas] = df_dummy[cols_exogenas].shift(24)

# Creamos una variable nueva: "El precio que hubo hace 24 horas". 
df_dummy['price_lag24'] = df_dummy['price'].shift(24)
# Suele ser el predictor más potente en mercados eléctricos.

# 5. SACRIFICAMOS EL PRIMER DÍA: Empezamos explícitamente el 2 de enero de 2020
# Como 'datetime' es el índice, podemos filtrar usando strings de fecha directamente
df_dummy = df_dummy.loc['2020-01-02':] 

# Por seguridad, si hubiera algún otro NaN en el dataset original, lo eliminamos
df_dummy = df_dummy.dropna()

# =======================================================================================
# 1. PREPARACIÓN Y DIVISIÓN TEMPORAL
# =======================================================================================
X = df_dummy.drop(columns=['price'])
y = df_dummy['price']

# División cronológica estricta: entrenar con 2020-2023, validar con 2024 completo
X_train, X_test = X[X['year'] < 2024], X[X['year'] == 2024]
y_train, y_test = y[X['year'] < 2024], y[X['year'] == 2024]

# Eliminamos 'year' de los predictores
X_train = X_train.drop(columns=['year'])
X_test = X_test.drop(columns=['year'])

# Variables categóricas temporales identificadas
cat_features = ['hour', 'month', 'dayofweek', 'is_weekend']
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')


In [3]:
# ==============================
# 5. Elegir los órdenes p y q del modelo SARIMA
# ==============================
# En Box-Jenkins a partir de los gráficos de ACF y PACF, elegimos p y q
# En nuestro caso fueron p=2 y q=1
# En ARIMA, el orden es (p, d, q).
# El valor d e sel numero de diferenciaciones para hacer la serie estacionaria,
#  pero como la serie ya es estacionaria, no es necesario diferenciar (d=0)
#  aunque se puede poner d=1 sin que cambie el resultado matemático porque
#  ARIMA con d=1 en una serie ya estacionaria es equivalente a ARMA con d=0
p, d, q = 2, 1, 1           # parte no estacional (igual que ARIMA/ARMA)
P, D, Q, s = 2, 1, 0, 24    # parte estacional 


modelo_sarimax = SARIMAX(
    y_train,
    exog=X_train,
    order=(p, d, q),
    seasonal_order=(P, D, Q, s),
    enforce_stationarity=False,
    enforce_invertibility=False
)


# Ajustar el modelo
modelo_ajustado = modelo_sarimax.fit(low_memory=True)  # Reduce uso de memoria

# Mostrar un resumen del modelo
print(modelo_ajustado.summary())


# ==============================
# 5.2. Verificar residuos (validación)
# ==============================

# Verificar residuos (validación)
residuos = modelo_ajustado.resid

# Gráfico ACF de residuos
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
plot_acf(residuos, ax=axes[0], lags=96)
axes[0].set_title('ACF de residuos del modelo SARIMA')

plot_pacf(residuos, ax=axes[1], lags=96)
axes[1].set_title('PACF de residuos del modelo SARIMA')
plt.savefig('sarima_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

# Prueba de Ljung-Box (si p-value > 0.05 los residuos son ruido blanco)
lb_test = acorr_ljungbox(residuos, lags=[10], return_df=True)
print(f"\nTEST LJUNG-BOX")
print(lb_test.to_string())
if (lb_test['lb_pvalue'] > 0.05).all():
    print("los residuos son ruido blanco")
else:
    print("los residuos NO son ruido blanco (tienen estructura/autocorrelacion)")

# ==============================
# 6. Predecir los valores del conjunto de prueba
# ==============================
# forecast recibe el número de pasos a predecir y las variables exógenas del conjunto de prueba
predicciones = modelo_ajustado.forecast(steps=len(y_test), exog=X_test)
predicciones.index = y_test.index   # para que luego se pueda gaficar bien

# ==============================
# 7. AMPLIADA - Métricas de precisión
# ==============================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

mae  = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
mape = mean_absolute_percentage_error(y_test, predicciones) * 100

# Métrica adicional: R² (qué % de varianza explica el modelo)
ss_res = np.sum((y_test - predicciones) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot)

# Naive benchmark: predecir siempre el valor anterior (lag-1)
naive_pred = y_test.shift(1).dropna()
rmse_naive = np.sqrt(mean_squared_error(y_test[1:], naive_pred))

print("\n========== MÉTRICAS DE PRECISIÓN ==========")
print(f"  MAE   (Error Absoluto Medio):        {mae:.4f} €/MWh")
print(f"  RMSE  (Raíz Error Cuadrático Medio): {rmse:.4f} €/MWh")
print(f"  MAPE  (Error Porcentual Medio):       {mape:.2f}%")
print(f"  R²    (Coeficiente determinación):    {r2:.4f}")
print(f"  RMSE modelo naive (benchmark):        {rmse_naive:.4f} €/MWh")
print(f"  El modelo es {'MEJOR' if rmse < rmse_naive else 'PEOR'} que el naive")
print("============================================\n")


# ==============================
# 8. CORREGIDA - Gráficas con zoom y separadas
# ==============================
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Gráfica 1: visión global (últimas 4 semanas de train + todo el test) ---
CONTEXTO = 672  # horas de entrenamiento que se muestran como contexto

ax1 = axes[0]
ax1.plot(y_train[-CONTEXTO:].index, y_train[-CONTEXTO:],
         label='Entrenamiento (contexto)', color='steelblue', alpha=0.7)
ax1.plot(y_test.index, y_test,
         label='Real (prueba)', color='blue', linewidth=1.5)
ax1.plot(y_test.index, predicciones,
         label='Predicciones SARIMA', color='red', linestyle='--', linewidth=1.5)
ax1.axvline(x=y_test.index[0], color='gray', linestyle=':', linewidth=1.5,
            label='Inicio prueba')
ax1.set_title('Vista general: últimos meses de train + período de prueba')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Precio (€/MWh)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Gráfica 2: zoom solo en el período de prueba ---
ax2 = axes[1]
ax2.plot(y_test.index, y_test,
         label='Real', color='blue', linewidth=1.5)
ax2.plot(y_test.index, predicciones,
         label='Predicciones SARIMA', color='red', linestyle='--', linewidth=1.5)
ax2.fill_between(y_test.index,
                 predicciones - rmse,
                 predicciones + rmse,
                 alpha=0.15, color='red', label=f'±1 RMSE ({rmse:.2f})')
ax2.set_title(f'Zoom período de prueba — MAPE: {mape:.2f}% | RMSE: {rmse:.2f} | R²: {r2:.3f}')
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Precio (€/MWh)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sarima_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

# =======================================================================================
# 9. EXPORTAR Y GUARDAR RESULTADOS (MODELO SARIMA)
# =======================================================================================
# Creamos un nuevo dataframe para comparar la realidad vs la predicción de SARIMA
df_resultados_sarima = pd.DataFrame({
    'Precio_Real': y_test,
    'Prediccion_SARIMA': predicciones,
    'Diferencia_SARIMA': y_test - predicciones
}, index=y_test.index)

# Añadimos variables temporales al dataframe de resultados para agrupar
df_resultados_sarima['hour'] = df_resultados_sarima.index.hour
df_resultados_sarima['month'] = df_resultados_sarima.index.month

# Calculamos el MAE promedio por hora del día para SARIMA
mae_por_hora_sarima = df_resultados_sarima.groupby('hour').apply(
    lambda x: mean_absolute_error(x['Precio_Real'], x['Prediccion_SARIMA'])
)

print("=== MAE DE SARIMA POR HORA DEL DÍA ===")
print(mae_por_hora_sarima)

# Guardamos los resultados en un archivo CSV específico para SARIMA
df_resultados_sarima.to_csv('predicciones_sarima_2024.csv')
print("\n¡Archivo 'predicciones_sarima_2024.csv' generado con éxito!")

print(f"Desviación Estándar Real (2024): {df_resultados_sarima['Precio_Real'].std():.2f}")
print(f"Desviación Estándar Predicha SARIMA: {df_resultados_sarima['Prediccion_SARIMA'].std():.2f}")

C:\Users\NITROPC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\Users\NITROPC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\Users\NITROPC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


KeyboardInterrupt: 

Que SARIMAX lleve 150 minutos (2 horas y media) significan que el modelo ha estado atrapado en un bucle infinito de optimización del que no iba a salir con buenos resultados.
El último mensaje (ConvergenceWarning: Maximum Likelihood optimization failed to converge) es la confirmación oficial de que el algoritmo tiró la toalla [C:\Users\NITROPC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\statespace\mlemodel.py:737]. Intentó ajustar las matemáticas de 31 variables exógenas más la estacionalidad en 43,000 filas y no pudo converger.
Detén la ejecución de tu script (puedes pulsar Ctrl + C en la terminal o el botón de stop en VS Code / Jupyter). No dejes que siga consumiendo recursos de tu NITROPC, porque los resultados que te va a dar tras este aviso no serán fiables.
Aquí tienes los 3 cambios urgentes que debes aplicar para solucionar esto y hacer que el modelo corra en menos de 5 minutos:
## 1. El truco del optimizador (Pasa de 150 minutos a 3 minutos)
Por defecto, SARIMAX utiliza un método de optimización muy estricto llamado L-BFGS-B. Con tantas variables exógenas, se satura. Cambia tu método de ajuste por method='powell', el cual es muchísimo más robusto para modelos con alta dimensionalidad (muchas columnas) y converge increíblemente rápido.
Modifica tu línea del .fit() para que quede exactamente así:

# Cambia tu línea actual por esta:modelo_ajustado = modelo_sarimax.fit(method='powell', low_memory=True)

## 2. Reduce las variables exógenas (X_train)
Tienes un problema severo de multicolinealidad. Le estás pasando al modelo el precio (price), la demanda, variables meteorológicas, tipos de generación, y además componentes de fecha manuales (year, month, day, dayofweek, hour).

* SARIMAX ya calcula internamente el tiempo a través de sus parámetros order y seasonal_order.
* Quita de tus variables X_train y X_test las columnas: year, month, day, dayofweek, hour e is_weekend. Dejar estas columnas confunde por completo al estimador estadístico.

## 3. Acorta el histórico de entrenamiento
Para SARIMAX, entrenar con 5 años de datos horarios es contraproducente. Los patrones del mercado eléctrico de hace 5 años ya no se parecen a los de 2024.

* Entrena al modelo solo con el último año o los últimos 6 meses de datos en tu y_train.
* Esto reducirá tus filas de 43,000 a unas 8,760 (o menos), haciendo que el modelo vuele y capture la estacionalidad reciente con mayor precisión.

Si aplicas el cambio de method='powell' y recortas las columnas temporales del dataset exógeno, ¿puedes verificar si el script logra terminar de ejecutarse en pocos minutos y si finalmente te genera el archivo .csv?